# Principal Component Analysis — from four customers to four features

This notebook follows the video's calculations, then lets you change the
component count and answer six short tasks. Run cells from top to bottom.
The 500 customers are **synthetic**, generated here with a fixed seed; no
download or course-repository import is needed. The coupon outcome is used
only for the final prediction check, never to fit PCA.

**Requirements:** Python 3.10+, NumPy, and scikit-learn. In Google Colab,
choose **Runtime → Run all**. Locally, install them with
`pip install numpy scikit-learn jupyter` and open this file in Jupyter.

The practice checks show ✅ for a correct answer, ❌ with the expected
value for an incorrect answer, and ⬜ when you have not answered yet. An
untouched copy runs to the end. Work through it before opening the separate
solution notebook.

In [1]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def check(label, name, expected, *, atol=1e-6):
    if name not in globals() or globals()[name] is None:
        print(f"⬜ {label}: create {name} and run this cell again")
        return
    got = globals()[name]
    try:
        correct = bool(np.allclose(got, expected, rtol=1e-6, atol=atol))
    except (TypeError, ValueError):
        correct = False
    if correct:
        print(f"✅ {label}")
    else:
        print(f"❌ {label}: expected {np.asarray(expected)}, got {got}")

HERO = np.array([[2., 5., 120., 30.], [3., 4., 130., 32.],
                 [5., 8., 240., 40.], [6., 7., 250., 42.]])
HERO_RESPONSE = np.array([0, 1, 0, 1])  # B and D used the store coupon.
rng = np.random.default_rng(10)
n = 496
activity = rng.normal(0., 1., n)
preference = rng.normal(0., 1., n)
store = 4. + 1.3*activity + .7*preference + rng.normal(0, .2, n)
app = 6. + 1.3*activity - .7*preference + rng.normal(0, .2, n)
spend = 185. + 40.*activity + rng.normal(0, 6., n)
order_value = 36. + activity + 4.*preference + rng.normal(0, 1.2, n)
used_coupon = (preference + rng.normal(0, .3, n) > 0).astype(int)
X = np.vstack((HERO, np.column_stack((store, app, spend, order_value))))
y = np.concatenate((HERO_RESPONSE, used_coupon))
X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400], y[400:]
assert X.shape == (500, 4) and X_test.shape == (100, 4)
assert int(y_test.sum()) == 50
print('400 training customers; 100 held-out customers; 50 held-out coupon users')

400 training customers; 100 held-out customers; 50 held-out coupon users


## Layer 1 — Follow the video

### Four customers: center, find directions, and project

Start with store visits and app sessions. Subtract each column's mean so a
score measures displacement from the average customer. The covariance
matrix records the variance of each centered column on its diagonal and
their joint variation off the diagonal.

In [2]:
hero = HERO[:, :2]
mean = hero.mean(axis=0)
centered = hero - mean
covariance = centered.T @ centered / (len(hero) - 1)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
components = eigenvectors[:, order].T

# An eigenvector and its negative describe the same component line. Align
# signs with the directions derived by hand in the video: PC1=(1,1)/√2,
# PC2=(1,-1)/√2. NumPy may return the opposite PC2 direction.
video_directions = np.array([[1., 1.], [1., -1.]]) / np.sqrt(2.)
components *= np.where(np.sum(components * video_directions, axis=1) < 0,
                       -1., 1.)[:, None]
hero_scores = centered @ components.T
squared_spread = np.sum(hero_scores**2, axis=0)
print('mean:', mean)
print('covariance:\n', covariance)
print('component rows:\n', np.round(components, 3))
print('A–D scores:\n', np.round(hero_scores, 3))
print('squared spread:', squared_spread,
      'variance shares:', np.round(eigenvalues/eigenvalues.sum(), 3))
np.testing.assert_allclose(mean, [4., 6.])
np.testing.assert_allclose(covariance, [[10/3, 8/3], [8/3, 10/3]])
np.testing.assert_allclose(eigenvalues, [6., 2/3])
np.testing.assert_allclose(squared_spread, [18., 2.])
np.testing.assert_allclose(hero_scores[:, 1],
                           [-1/np.sqrt(2), 1/np.sqrt(2),
                            -1/np.sqrt(2), 1/np.sqrt(2)])

mean: [4. 6.]
covariance:
 [[3.33333333 2.66666667]
 [2.66666667 3.33333333]]
component rows:
 [[ 0.707  0.707]
 [ 0.707 -0.707]]
A–D scores:
 [[-2.121 -0.707]
 [-2.121  0.707]
 [ 2.121 -0.707]
 [ 2.121  0.707]]
squared spread: [18.  2.] variance shares: [0.9 0.1]


Scikit-learn gives the same *lines* on these same four rows. Its component
signs may also differ, so align each library direction before comparing
the numbers. Flipping a component flips its scores but changes neither its
explained variance nor any reconstruction.

In [3]:
library_hero = PCA(n_components=2, svd_solver='full').fit(hero)
library_signs = np.where(np.sum(library_hero.components_ * components,
                                axis=1) < 0, -1., 1.)
library_scores = library_hero.transform(hero) * library_signs
np.testing.assert_allclose(library_hero.components_ * library_signs[:, None],
                           components)
np.testing.assert_allclose(library_scores, hero_scores)
np.testing.assert_allclose(library_hero.explained_variance_ratio_, [.9, .1])
print('NumPy and scikit-learn agree on both component lines and all four scores.')

NumPy and scikit-learn agree on both component lines and all four scores.


### A new customer: transform and reconstruct

The new row is (7 store visits, 8 app sessions). First center it using the
**training mean**. One PC1 score rebuilds (6.5, 8.5); keeping PC2 as well
rebuilds (7, 8) exactly. This is the calculation shown in the video.

In [4]:
new_customer = np.array([7., 8.])
new_scores = (new_customer - mean) @ components.T
rebuilt_one = mean + new_scores[0] * components[0]
rebuilt_two = mean + new_scores @ components
print('new customer scores:', np.round(new_scores, 3))
print('one-component reconstruction:', rebuilt_one)
print('two-component reconstruction:', rebuilt_two)
np.testing.assert_allclose(new_scores, [5/np.sqrt(2), 1/np.sqrt(2)])
np.testing.assert_allclose(rebuilt_one, [6.5, 8.5])
np.testing.assert_allclose(rebuilt_two, new_customer)

new customer scores: [3.536 0.707]
one-component reconstruction: [6.5 8.5]
two-component reconstruction: [7. 8.]


### Four features: standardise, then choose a count

Raw monthly spend has much larger numerical variation than weekly visits.
Without standardisation, PC1 mostly measures spend. `StandardScaler` fits
only on the 400 training customers; the same fitted scaler transforms the
held-out 100. Scikit-learn's `PCA` centers columns but does not standardise
them for you.

In [5]:
raw_pca = PCA(n_components=4, svd_solver='full').fit(X_train)
scaler = StandardScaler().fit(X_train)
train_scaled = scaler.transform(X_train)
test_scaled = scaler.transform(X_test)
scaled_pca = PCA(n_components=4, svd_solver='full').fit(train_scaled)
shares = scaled_pca.explained_variance_ratio_
cumulative = np.cumsum(shares)
features = ('store visits', 'app sessions', 'monthly spend', 'order value')
for name, raw_weight, scaled_weight in zip(
        features, raw_pca.components_[0], scaled_pca.components_[0]):
    print(f'{name:16} raw {raw_weight:+.3f}   standardised {scaled_weight:+.3f}')
print('raw PC1 variance share:', round(raw_pca.explained_variance_ratio_[0], 4))
print('standardised component shares:', np.round(shares, 4))
print('cumulative shares:', np.round(cumulative, 4))
np.testing.assert_allclose(raw_pca.explained_variance_ratio_[0], .9879, atol=.0001)
np.testing.assert_allclose(shares[:2], [.6407, .3433], atol=.0001)
np.testing.assert_allclose(cumulative[1], .9840, atol=.0001)

store visits     raw +0.031   standardised +0.581
app sessions     raw +0.033   standardised +0.486
monthly spend    raw +0.999   standardised +0.611
order value      raw +0.022   standardised +0.231
raw PC1 variance share: 0.9879
standardised component shares: [0.6407 0.3433 0.0106 0.0055]
cumulative shares: [0.6407 0.984  0.9945 1.    ]


In [6]:
minimum_count = int(np.searchsorted(cumulative, .95) + 1)
reconstruction_errors = []
for count in range(5):
    if count == 0:
        rebuilt = np.zeros_like(test_scaled)  # training mean in scaled units
    else:
        fitted = PCA(n_components=count, svd_solver='full').fit(train_scaled)
        rebuilt = fitted.inverse_transform(fitted.transform(test_scaled))
    reconstruction_errors.append(
        float(np.mean(np.sum((test_scaled - rebuilt)**2, axis=1))))
print('minimum components for this example’s 95% requirement:', minimum_count)
print('held-out reconstruction error, 0–4 components:',
      np.round(reconstruction_errors, 3))
assert minimum_count == 2
np.testing.assert_allclose(reconstruction_errors,
                           [4.55, 1.30, .06, .03, 0.], atol=.005)

minimum components for this example’s 95% requirement: 2
held-out reconstruction error, 0–4 components: [4.548 1.295 0.061 0.027 0.   ]


### Variance does not use the coupon label

The PCA fit above never sees `y`. A logistic regression trained on PC1 alone
gets 55 of 100 held-out coupon outcomes right; PC1 and PC2 together get 92.
Keeping 90% of variance on the four hand-worked rows is a different result
from keeping 64.1% with PC1 on the 500-customer, four-feature dataset.

In [7]:
train_scores = scaled_pca.transform(train_scaled)
test_scores = scaled_pca.transform(test_scaled)
correct_counts = []
for count in (1, 2, 3):
    model = LogisticRegression().fit(train_scores[:, :count], y_train)
    correct_counts.append(int(np.sum(
        model.predict(test_scores[:, :count]) == y_test)))
original = make_pipeline(StandardScaler(), LogisticRegression()).fit(
    X_train, y_train)
original_correct = int(np.sum(original.predict(X_test) == y_test))
print('correct / 100 for PC1, PC1+PC2, PC1+PC2+PC3:', correct_counts)
print('correct / 100 for all four original columns:', original_correct)
assert correct_counts == [55, 92, 91] and original_correct == 91

correct / 100 for PC1, PC1+PC2, PC1+PC2+PC3: [55, 92, 91]
correct / 100 for all four original columns: 91


PCA's directions are straight. For the separate curved app-use example in
the video, the first straight component retains 93.3% of the standardised
variance but still leaves a visible pattern in the reconstruction gaps.

In [8]:
months = np.linspace(1., 24., 21)
curved = np.column_stack((months, 1. + 9. * (1. - np.exp(-months / 5.))))
curved_scaled = StandardScaler().fit_transform(curved)
curved_share = PCA(n_components=2).fit(curved_scaled).explained_variance_ratio_[0]
print('curved example, PC1 variance share:', f'{curved_share:.1%}')
np.testing.assert_allclose(curved_share, .9332, atol=.0001)
print('Worked calculations passed.')

curved example, PC1 variance share: 93.3%
Worked calculations passed.


## Layer 2 — Experiment with the operating choice

Predict first: will using two components instead of one improve coupon
accuracy in both the raw and standardised runs? The table below refits PCA
and logistic regression on the *same training rows* for each choice. Its
reconstruction errors are measured in the units of each run, so compare
errors **within** a raw or standardised group, not across the groups.

Change `component_counts` or add a `standardise` choice, then rerun this
cell. Which choice keeps useful coupon information, and what does the
variance share alone fail to tell you?

In [9]:
component_counts = (1, 2, 3)
print('standardise  components  variance kept  test error  coupon correct/100')
for standardise in (False, True):
    if standardise:
        fit_rows, test_rows = train_scaled, test_scaled
    else:
        fit_rows, test_rows = X_train, X_test
    for count in component_counts:
        fitted = PCA(n_components=count, svd_solver='full').fit(fit_rows)
        fit_scores = fitted.transform(fit_rows)
        held_out_scores = fitted.transform(test_rows)
        error = np.mean(np.sum((test_rows - fitted.inverse_transform(
            held_out_scores))**2, axis=1))
        classifier = LogisticRegression().fit(fit_scores, y_train)
        correct = int(np.sum(classifier.predict(held_out_scores) == y_test))
        print(f'{str(standardise):11} {count:10} '
              f'{fitted.explained_variance_ratio_.sum():13.1%} '
              f'{error:11.3f} {correct:12}/100')

standardise  components  variance kept  test error  coupon correct/100
False                1         98.8%      16.718           47/100
False                2        100.0%       0.250           93/100
False                3        100.0%       0.152           92/100
True                 1         64.1%       1.295           55/100
True                 2         98.4%       0.061           92/100
True                 3         99.5%       0.027           91/100


## Layer 3 — Your turn

Fill each answer cell, then run the check beneath it. The checks do not
reveal your answer until you create the requested variable. If you make a
mistake, the check prints the expected value so you can inspect the step.
The solution notebook is separate so you can try these first.

**1. Center customer A.** Create `a_centered`, an array with A's store
visits and app sessions after subtracting the two-column mean.

In [10]:
a_centered = hero[0] - mean

In [11]:
check('A centered', 'a_centered', [-2., -1.])

✅ A centered


**2. Compute A's PC1 score.** Create `a_pc1_score` using the centered
values and the video-aligned PC1 direction in `components[0]`.

In [12]:
a_pc1_score = float(a_centered @ components[0])

In [13]:
check('A PC1 score', 'a_pc1_score', -3/np.sqrt(2))

✅ A PC1 score


**3. Rebuild a new customer from one score.** Create
`one_component_new_row`, an array rebuilt from the training mean, the new
row's PC1 score, and the PC1 direction. Start from `new_scores[0]`.

In [14]:
one_component_new_row = mean + new_scores[0] * components[0]

In [15]:
check('one-component reconstruction', 'one_component_new_row', [6.5, 8.5])

✅ one-component reconstruction


**4. Choose a component count.** Create `student_count`: the smallest
number of standardised components whose *cumulative* training variance
share meets the example requirement of at least 95%. Use `cumulative`.

In [16]:
student_count = int(np.searchsorted(cumulative, .95) + 1)

In [17]:
check('95% component count', 'student_count', 2)

✅ 95% component count


**5. Measure the coupon gain.** Create `coupon_gain`, the improvement in
held-out accuracy from using PC1+PC2 instead of PC1 alone. Express it as
a proportion, not a percentage number. Use `correct_counts` and 100 rows.

In [18]:
coupon_gain = (correct_counts[1] - correct_counts[0]) / len(y_test)

In [19]:
check('coupon accuracy gain', 'coupon_gain', .37)

✅ coupon accuracy gain


**6. Transform new rows yourself.** Define `transform_rows(rows, mean,
components)` using centering and matrix multiplication. It should return
both PCA scores for each row. The check uses customer A and the new (7, 8)
customer and accepts either list or NumPy array output.

In [20]:
def transform_rows(rows, mean, components):
    return (np.asarray(rows) - mean) @ components.T

In [21]:
if 'transform_rows' not in globals():
    print('⬜ transform rows: define transform_rows and run this cell again')
else:
    try:
        transformed = transform_rows(np.array([[2., 5.], [7., 8.]]),
                                     mean, components)
    except Exception as error:
        print(f'❌ transform rows: {type(error).__name__}: {error}')
    else:
        expected = np.array([[-3/np.sqrt(2), -1/np.sqrt(2)],
                             [5/np.sqrt(2), 1/np.sqrt(2)]])
        try:
            correct = np.shape(transformed) == (2, 2) and np.allclose(
                transformed, expected, rtol=1e-6, atol=1e-6)
        except (TypeError, ValueError):
            correct = False
        print('✅ transform rows' if correct else
              f'❌ transform rows: expected {np.round(expected, 3)}, got {transformed}')

✅ transform rows


### Explain the result in your own words

1. Why does monthly spend dominate raw PC1, even though the four features
   were generated from only two underlying customer quantities?
2. Why does PC2 help the coupon classifier more than its variance share
   might suggest?
3. Which operations must be fitted on training rows only before new rows
   can be transformed?
4. Why can NumPy print the negative of the video’s PC2 direction without
   changing the PCA solution?

### Reference explanations

1. Monthly spend is measured in dollars and has much larger raw variance.
   PCA maximises numerical variance, so standardise first when that unit
   difference should not determine the direction.
2. PCA never uses the coupon label. The preference that predicts coupon
   use lies mostly along PC2, even though PC2 holds less overall variance.
3. Fit the scaler, PCA directions, and classifier using training rows only;
   then apply those fitted objects to held-out or new rows.
4. A component and its negative describe the same line. All associated
   scores change sign, while variance and reconstructed points stay fixed.